In [1]:
import numpy as np
import pandas as pd
from narwhals import Int64
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt

# 1. Load Data

In [2]:
train_df = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index_col=0)

# 2. Preprocess

In [3]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")

# Defining categorical columns
cat_cols = ["VehicleStyle", "VehicleColor", "SubjectRace", "SubjectEyeColor", "SubjectHairColor", "Zip", "StopDescription", "District"]
# Defining discrete columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "isLA","VehicleYear", "SubjectHasPhotoID", "hour", "night", "month", "weekend", "isDriver"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "month", "hour"]
bin_cols = ["SubjectGender", "isLA", "SubjectHasPhotoID", "weekend", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)

target = "search_occurred"
X_train = train_df.drop([target], axis = 1)
y_train = train_df[target]

# 3. Define classes and pipeline function

In [4]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self

def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    ordinal = ColumnTransformer([
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan),cat_cols)], remainder="passthrough", verbose_feature_names_out=False)

    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10), skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("ordinal", ordinal),
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    pipeline[:-2].set_output(transform="pandas")
    return pipeline

# 3a) Define pipeline

In [5]:
pipeline = build_pipeline(DecisionTreeClassifier(random_state=10), cat_cols, bin_cols, scale_cols)

# 4. Define grid search parameters and run grid search cv
## gini/entropy, max_depth, min_samples_split, min_samples_leaf

In [7]:
param_grid = [{"model__criterion":["gini", "entropy"], "model__max_depth":[None, 3, 6, 10], "model__min_samples_split":[2, 4, 6], "model__min_samples_leaf":[1, 3, 5]}]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

grid_search = GridSearchCV(cv = cv, estimator = pipeline, param_grid=param_grid, scoring=["accuracy", "balanced_accuracy", "precision", "recall", "roc_auc", "f1", "neg_brier_score"], refit = "balanced_accuracy", verbose=2)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2; total time=  10.3s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2; total time=  10.6s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2; total time=   9.5s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2; total time=   9.7s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=2; total time=  10.1s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=4; total time=   8.0s
[CV] END model__criterion=gini, model__max_depth=None, model__min_samples_leaf=1, model__min_samples_split=4; total time=   8.2s
[CV] END model__criterion=gini, mod

C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[CV] END model__criterion=gini, model__max_depth=3, model__min_samples_leaf=1, model__min_samples_split=2; total time=   1.5s


C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[CV] END model__criterion=gini, model__max_depth=3, model__min_samples_leaf=1, model__min_samples_split=2; total time=   1.4s


C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


... (172 outputs left out)

[CV] END model__criterion=entropy, model__max_depth=3, model__min_samples_leaf=5, model__min_samples_split=6; total time=   2.2s


C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[CV] END model__criterion=entropy, model__max_depth=3, model__min_samples_leaf=5, model__min_samples_split=6; total time=   2.7s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=2; total time=   3.5s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=2; total time=   3.1s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=2; total time=   3.6s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=2; total time=   3.1s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=2; total time=   3.8s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samples_split=4; total time=   4.2s
[CV] END model__criterion=entropy, model__max_depth=6, model__min_samples_leaf=1, model__min_samp

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(memo...m_state=10))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 3, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 4, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['accuracy', 'balanced_accuracy', ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'balanced_accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=

In [8]:
results = pd.DataFrame(grid_search.cv_results_)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,params,split0_test_accuracy,...,std_test_f1,rank_test_f1,split0_test_neg_brier_score,split1_test_neg_brier_score,split2_test_neg_brier_score,split3_test_neg_brier_score,split4_test_neg_brier_score,mean_test_neg_brier_score,std_test_neg_brier_score,rank_test_neg_brier_score
0,9.295512,0.400962,0.831669,0.140102,gini,None,1,2,"{'model__criterion': 'gini', 'model__max_depth...",0.905745,...,0.010485,6,-0.094265,-0.095481,-0.095678,-0.094104,-0.092177,-0.094341,0.001251,72
1,7.672852,0.813469,0.829810,0.053546,gini,None,1,4,"{'model__criterion': 'gini', 'model__max_depth...",0.911556,...,0.012552,9,-0.087431,-0.088439,-0.088501,-0.085551,-0.086548,-0.087294,0.001130,69
2,7.437377,0.275372,0.784225,0.068482,gini,None,1,6,"{'model__criterion': 'gini', 'model__max_depth...",0.914070,...,0.014173,5,-0.082789,-0.081801,-0.083215,-0.079529,-0.080899,-0.081647,0.001329,67
3,7.059302,0.291902,0.776808,0.138941,gini,None,3,2,"{'model__criterion': 'gini', 'model__max_depth...",0.920747,...,0.008543,10,-0.070594,-0.069982,-0.071446,-0.069131,-0.069745,-0.070179,0.000788,61
4,8.435983,0.933431,0.838908,0.219571,gini,None,3,4,"{'model__criterion': 'gini', 'model__max_depth...",0.920747,...,0.008543,10,-0.070594,-0.069982,-0.071446,-0.069131,-0.069745,-0.070179,0.000788,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2.301320,0.134156,0.824042,0.028129,entropy,10,3,4,"{'model__criterion': 'entropy', 'model__max_de...",0.943043,...,0.013058,31,-0.048091,-0.047286,-0.049241,-0.047225,-0.047815,-0.047931,0.000731,40
68,2.658134,0.837396,1.165261,0.339768,entropy,10,3,6,"{'model__criterion': 'entropy', 'model__max_de...",0.943043,...,0.013058,31,-0.048091,-0.047286,-0.049241,-0.047225,-0.047815,-0.047931,0.000731,40
69,3.497585,1.080394,1.105134,0.237743,entropy,10,5,2,"{'model__criterion': 'entropy', 'model__max_de...",0.943373,...,0.015422,28,-0.047916,-0.046775,-0.049039,-0.046931,-0.047756,-0.047684,0.000811,37
70,4.480225,0.268432,1.140117,0.108567,entropy,10,5,4,"{'model__criterion': 'entropy', 'model__max_de...",0.943373,...,0.015422,28,-0.047916,-0.046775,-0.049039,-0.046931,-0.047756,-0.047684,0.000811,37


In [14]:
# Best parameters based on:
## Balanced accuracy
print("Balanced accuracy:", results["params"][results["rank_test_balanced_accuracy"] == 1].item())
## Precision
print("Precision:", results["params"][results["rank_test_precision"] == 1].head(1).item())
## Recall
print("Recall:", results["params"][results["rank_test_recall"] == 1].head(1).item())
## F1
print("F1:", results["params"][results["rank_test_f1"] == 1].head(1).item())
## AUC
print("AUC:", results["params"][results["rank_test_roc_auc"] == 1].head(1).item())
## Brier
print("Brier score:", results["params"][results["rank_test_neg_brier_score"] == 1].head(1).item())

Balanced accuracy: {'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Precision: {'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 3, 'model__min_samples_split': 2}
Recall: {'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
F1: {'model__criterion': 'entropy', 'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__min_samples_split': 2}
AUC: {'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 3, 'model__min_samples_split': 2}
Brier score: {'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
